In [14]:
from wavefield_computation import setup_model_and_geometry
from examples.seismic.datasets import SeismogramDataset, VelocityModel
import segyio
import numpy as np

def save_to_segy(velocity_model, output_path):
    """
    Save the velocity model to SEGY format with proper CDP headers.
    
    Parameters:
    velocity_model (VelocityModel): Instance of your VelocityModel class
    output_path (str): Path to save the SEGY file
    """
    # Ensure the model is loaded and ready
    velocity_model._ensure_model_ready()
    
    # Get model dimensions and data
    n_traces = len(velocity_model.x)
    n_samples = len(velocity_model.z)
    vp_data = velocity_model.vp.T  # Transpose to get (traces, samples)
    
    # Create SEGY file
    spec = segyio.spec()
    spec.format = 5  # IEEE floating point
    spec.sorting = 2  # CDP sorting
    spec.samples = velocity_model.z.astype(np.float32)  # Depth/Z axis
    spec.ilines = np.arange(n_traces) + 1  # Trace numbers
    spec.tracecount = n_traces
    with segyio.create(output_path, spec) as segyfile:
        # Write the data
        segyfile.trace[:] = vp_data.astype(np.float32)
        
        # Set headers
        for i, trace in enumerate(segyfile.header):
            # CDP number - just use sequential numbering
            trace.update({
                segyio.TraceField.TRACE_SEQUENCE_LINE: i + 1,
                segyio.TraceField.TRACE_SEQUENCE_FILE: i + 1,
                segyio.TraceField.FieldRecord: 1,  # Could be survey/line number
                segyio.TraceField.TraceNumber: i + 1,
                segyio.TraceField.CDP: i + 1,  # CDP number
                segyio.TraceField.CDP_TRACE: 1,  # Always 1 for 2D
                segyio.TraceField.offset: 1,  # Minimum offset
                segyio.TraceField.SourceX: int(velocity_model.x[i] * 100),
                segyio.TraceField.GroupX: int(velocity_model.x[i] * 100),
                segyio.TraceField.CDP_X: int(velocity_model.x[i] * 100),  # Scaled to cm
                segyio.TraceField.CDP_Y: 0,  # For 2D this is typically 0
                # segyio.TraceField.ns: n_samples,  # Number of samples
                # segyio.TraceField.dt: int(velocity_model.dz * 1000),  # Sample interval in microseconds
            })
        
        # Update binary header
        segyfile.bin.update({
            segyio.BinField.Traces: n_traces,
            segyio.BinField.Samples: n_samples,
            segyio.BinField.Interval: int(velocity_model.dz * 1000),  # Sample interval in microseconds
        })

In [5]:
from config import *
dataset = SeismogramDataset(PATH_DATA, "sou", invert_elevs=True)
xmin, xmax = min(dataset.x_coords.min(), dataset.opposite_x.min()), max(dataset.x_coords.max(), dataset.opposite_x.max())
spacing = (0.025, 0.025)
velmodel = VelocityModel(
    PATH_MODEL,
    dx=spacing[0],
    dz=spacing[1],
    clip=True,
    xmin=xmin - 3,
    xmax=xmax + 3,
    zmin=-318,
)
velmodel.pad_left(4 + 2)
velmodel.pad_right(8 * int(0.5 / spacing[0]) + 2)
velmodel.pad_bottom(10 * int(0.5 / spacing[0]) + 2)
velmodel.pad_top(7 * int(0.5 / spacing[0]))

In [15]:
save_to_segy(velmodel, "conventional_velmodel.sgy")

/home/andrey/miniconda3/envs/devito/lib/python3.12/site-packages/segyio/utils.py:18: RuntimeWarning: Implicit conversion to contiguous array
  warnings.warn(msg, RuntimeWarning)
